# E012 — Audio CMVN vs CMN

Tests CMVN (Cepstral Mean **and Variance** Normalization) against the E008 +All
baseline that uses CMN (mean only). Everything else is identical to E008 +All:

- UBM 32 components, MAP r=16
- MFCC 13+Δ+ΔΔ
- +All augmentation (noise SNR=20dB + speed ±10%) on train, originals on val
- LOSO 3-fold, seed=67

**Only change:** after stacking `[mfcc, delta, delta2]`, divide by `std + 1e-10`.

E008 reference (+All, CMN):
- Fold 0=3.47%, Fold 1=8.33%, Fold 2=0.83%, Mean=4.21±3.11%, min-DCF=0.0509

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
CMVN_COLOR = "#E74C3C"
CMN_COLOR  = "#95A5A6"

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Feature extraction — CMVN

Identical to E008 except `extract_mfcc` now divides by `std + 1e-10` after
mean subtraction. All other helpers are unchanged.

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def aug_noise_audio(y: np.ndarray, snr_db: float = 20.0,
                    rng: np.random.Generator = None) -> np.ndarray:
    """Add white noise at target SNR (dB)."""
    signal_power = np.mean(y ** 2) + 1e-10
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y)).astype(y.dtype)
    return y + noise


def aug_speed(y: np.ndarray, rate_range=(0.9, 1.1),
              rng: np.random.Generator = None) -> np.ndarray:
    """Random time stretch without changing pitch."""
    rate = rng.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)


# ── CMVN: subtract mean, then divide by std + eps ──────────────────────────
def extract_mfcc(y: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """MFCC + delta + delta-delta + CMVN. Returns (T, 39)."""
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T   # (T, 39)
    mfcc  -= mfcc.mean(axis=0)                    # CMN
    mfcc  /= (mfcc.std(axis=0) + 1e-10)           # variance normalization
    return mfcc


def extract_audio_augmented(wav_path: Path, rng: np.random.Generator):
    """Load WAV and return list of (y, sr): original + noise + speed."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise_audio(y, snr_db=20.0, rng=rng), sr),
        (aug_speed(y, rate_range=(0.9, 1.1), rng=rng), sr),
    ]


def extract_batch_augmented(df: pd.DataFrame, data_dir: Path, seed: int):
    """Extract CMVN features for all samples with +All augmentation."""
    rng = np.random.default_rng(seed)
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in extract_audio_augmented(find_wav(row["stem"], data_dir), rng):
            mfcc = extract_mfcc(y_aug, sr)
            all_mfcc.append(mfcc)
            all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


def extract_batch_original(df: pd.DataFrame, data_dir: Path):
    """Extract CMVN features from original WAVs only (val fold)."""
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        y, sr = librosa.load(find_wav(row["stem"], data_dir), sr=None, mono=True)
        mfcc = extract_mfcc(y, sr)
        all_mfcc.append(mfcc)
        all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


print("CMVN helpers defined.")

## 2. UBM + MAP functions (identical to E008)

In [ ]:
def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


def score_wav(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    """Score a WAV file using adapted vs UBM LLR."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc  = extract_mfcc(y, sr)
    return float((adapted.score_samples(mfcc) - ubm.score_samples(mfcc)).mean())


print("UBM+MAP functions defined.")

## 3. LOSO cross-validation (CMVN + +All augmentation)

- Train: +All augmentation (original + noise + speed)
- Val: original WAVs only
- UBM: 32 components, non-target frames only
- MAP: r=16, target frames only

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

oof_scores   = np.full(len(manifest), np.nan)
fold_results = []

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]

    print(f"\n{'='*55}")
    print(f"Fold {fold_id}")
    print(f"  train: {len(train_df)} samples, val: {len(val_df)} samples")

    # Extract augmented train frames (CMVN applied inside extract_mfcc)
    X_train, y_train = extract_batch_augmented(train_df, DATA, seed=SEED + fold_id)
    X_nt = X_train[y_train == 0]
    X_t  = X_train[y_train == 1]
    print(f"  frames → target: {len(X_t)}, non-target: {len(X_nt)} (with +All aug)")

    # Train UBM on non-target, adapt on target
    ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
    adapted = map_adapt(ubm, X_t, r=MAP_R)

    # Score val on ORIGINAL WAVs
    for idx, row in val_df.iterrows():
        oof_scores[idx] = score_wav(find_wav(row["stem"], DATA), adapted, ubm)

    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    eer, _     = compute_eer(val_scores[val_labels == 1], val_scores[val_labels == 0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels == 1], val_scores[val_labels == 0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    print(f"  → EER={eer*100:.2f}%, min-DCF={min_dcf:.4f}")

print("\nCV complete.")

## 4. Results — comparison with E008 CMN

In [ ]:
eers   = [r["eer"] * 100  for r in fold_results]
dcfs   = [r["min_dcf"]    for r in fold_results]
mean_e = np.mean(eers)
std_e  = np.std(eers)
mean_d = np.mean(dcfs)

# E008 +All CMN reference
e008_eers = [3.47, 8.33, 0.83]
e008_mean = np.mean(e008_eers)
e008_std  = np.std(e008_eers)
e008_dcf  = 0.0509

print(f"{'Config':<15} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 68)
print(f"{'E008 CMN (+All)':<15} {e008_eers[0]:>8.2f} {e008_eers[1]:>8.2f} {e008_eers[2]:>8.2f} "
      f"{e008_mean:>8.2f} {e008_std:>8.2f} {e008_dcf:>9.4f}")
print(f"{'E012 CMVN (+All)':<15} {eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
      f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}")
print("-" * 68)

delta = mean_e - e008_mean
sign  = "+" if delta > 0 else ""
print(f"\nΔ EER (CMVN − CMN): {sign}{delta:.2f}%  ({'regression' if delta > 0 else 'improvement'})")

eer_oof, _    = compute_eer(oof_scores[y_all == 1], oof_scores[y_all == 0])
dcf_oof, thr  = compute_min_dcf(oof_scores[y_all == 1], oof_scores[y_all == 0])
print(f"OOF overall: EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 5. Visualizations

In [ ]:
# 2x2 score distributions: per fold + OOF overall
fold_data = []
for fold_id, _, val_idx in iter_folds_loso(manifest, seed=SEED):
    fold_data.append({
        "scores": oof_scores[val_idx],
        "labels": manifest.loc[val_idx, "label"].to_numpy(),
    })

bin_edges = np.linspace(np.nanmin(oof_scores), np.nanmax(oof_scores), 35)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_data)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l == 1], s[l == 0])
    ax.hist(s[l == 0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"],
            label="non-target", density=True)
    ax.hist(s[l == 1], bins=bin_edges, alpha=0.75, color=COLORS["target"],
            label="target", density=True)
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2,
               label=f"thresh={thr_f:.2f}")
    ax.set_title(f"Fold {i}  —  EER={eer_f*100:.1f}%  (CMVN)")
    ax.set_xlabel("Score (LLR)")
    ax.legend(fontsize=8)

ax = axes[3]
ax.hist(oof_scores[y_all == 0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"],
        label="non-target", density=True)
ax.hist(oof_scores[y_all == 1], bins=bin_edges, alpha=0.75, color=COLORS["target"],
        label="target", density=True)
ax.axvline(thr, color=COLORS["green"], ls="--", lw=2,
           label=f"OOF thresh={thr:.2f}")
ax.set_title(f"OOF overall  —  EER={eer_oof*100:.1f}%  (CMVN)")
ax.set_xlabel("Score (LLR)")
ax.legend(fontsize=8)

plt.suptitle("E012 — Score distributions, CMVN +All", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Per-fold bar chart: CMN vs CMVN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-fold grouped bars
ax = axes[0]
x = np.arange(3)
width = 0.35
bars_cmn  = ax.bar(x - width/2, e008_eers, width,
                   label="E008 CMN",  color=CMN_COLOR,  alpha=0.85)
bars_cmvn = ax.bar(x + width/2, eers,      width,
                   label="E012 CMVN", color=CMVN_COLOR, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER: CMN vs CMVN")
ax.legend(fontsize=9)

# Right: mean ± std
ax = axes[1]
labels_bar = ["E008 CMN", "E012 CMVN"]
means_bar  = [e008_mean, mean_e]
stds_bar   = [e008_std,  std_e]
colors_bar = [CMN_COLOR, CMVN_COLOR]
bars = ax.bar(range(2), means_bar, color=colors_bar, alpha=0.85,
              yerr=stds_bar, capsize=8, error_kw=dict(elinewidth=2))
for bar, m, s in zip(bars, means_bar, stds_bar):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.1,
            f"{m:.2f}\n±{s:.2f}", ha="center", fontsize=9)
best_idx = int(np.argmin(means_bar))
bars[best_idx].set_edgecolor("gold")
bars[best_idx].set_linewidth(3)
ax.annotate("★ best",
            xy=(best_idx, means_bar[best_idx] - stds_bar[best_idx] - 0.2),
            ha="center", fontsize=10, color="goldenrod", fontweight="bold")
ax.set_xticks(range(2))
ax.set_xticklabels(labels_bar, fontsize=10)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("Mean ± std — CMN vs CMVN")

plt.suptitle("E012 — CMVN vs CMN (+All augmentation, UBM-MAP 32)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# DET curve: CMVN (E012) vs CMN (E008 reference values only available as per-fold EERs)
# We plot the CMVN OOF DET curve and annotate the CMN EER point
ticks     = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos  = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

fig, ax = plt.subplots(figsize=(7, 6))

valid = ~np.isnan(oof_scores)
fpr, tpr, _ = roc_curve(y_all[valid], oof_scores[valid])
far_c = np.clip(fpr, 1e-4, 1 - 1e-4)
frr_c = np.clip(1 - tpr, 1e-4, 1 - 1e-4)
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=CMVN_COLOR, lw=2.5,
        label=f"E012 CMVN  EER={eer_oof*100:.1f}%")

# Mark E008 CMN operating point (OOF EER was 9.17%)
e008_oof_eer = 9.17 / 100
ax.scatter([scipy_norm.ppf(e008_oof_eer)], [scipy_norm.ppf(e008_oof_eer)],
           color=CMN_COLOR, s=80, zorder=5, label=f"E008 CMN  OOF EER=9.2%")

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curve — E012 CMVN vs E008 CMN (OOF)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nFinal summary:")
print(f"  E008 CMN  +All: mean EER={e008_mean:.2f}±{e008_std:.2f}%  min-DCF={e008_dcf:.4f}")
print(f"  E012 CMVN +All: mean EER={mean_e:.2f}±{std_e:.2f}%  min-DCF={mean_d:.4f}")
print(f"  Δ EER: {mean_e - e008_mean:+.2f}%")